In [1]:
#!pip install streamlit ultralytics git+https://github.com/openai/CLIP.git

In [2]:
%%writefile app.py
import streamlit as st
import cv2
from ultralytics import YOLO
import torch
import clip
from PIL import Image
import numpy as np
import time

# SETUP & MODELS
st.set_page_config(page_title="EV Market Intelligence", layout="wide")

@st.cache_resource
def load_models():
    detector = YOLO("yolo11n.pt")
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model, preprocess = clip.load("ViT-B/32", device=device)
    return detector, model, preprocess, device

detector, clip_model, preprocess, device = load_models()

brands = ["BYD", "Xiaomi", "NIO", "Tesla", "Toyota", "Volkswagen", "Bus", "Truck", "Other Car Brands"]

brand_descriptions = [f"a photo of a {b} car in urban traffic" for b in brands]

text_inputs = clip.tokenize(brand_descriptions).to(device)

# UI LAYOUT
st.title("Real-Time Automotive Intelligence Dashboard")
tab1, tab2 = st.tabs(["Live Traffic Feed", "Upload Analysis"])

def process_video(source):
    cap = cv2.VideoCapture(source)
    st_frame = st.empty()
    stop_button = st.button("Stop Stream")

    while cap.isOpened() and not stop_button:
        ret, frame = cap.read()
        if not ret: break

        # Detection
        results = detector.track(frame, persist=True, classes=[2], conf=0.4)

        if results[0].boxes:
            for box in results[0].boxes:
                # Get coordinates
                x1, y1, x2, y2 = map(int, box.xyxy[0])

                # THE CLASSIFICATION FIX
                # Crop the car and send it to CLIP
                try:
                    crop = Image.fromarray(cv2.cvtColor(frame[y1:y2, x1:x2], cv2.COLOR_BGR2RGB))
                    img_input = preprocess(crop).unsqueeze(0).to(device)

                    with torch.no_grad():
                        logits, _ = clip_model(img_input, text_inputs)
                        probs = logits.softmax(dim=-1).cpu().numpy()

                    # Get the brand name and confidence
                    brand_idx = probs.argmax()
                    brand_name = brands[brand_idx]
                    confidence = probs[0][brand_idx]

                    # Draw Box and Label
                    color = (0, 255, 0) # Green
                    if probs.max() > 0.60:  # Only show the label if CLIP is reasonably confident
                      label = f"{brand_name} {confidence:.2f}"
                      cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
                      cv2.putText(frame, label, (x1, y1-10), cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2)
                except Exception as e:
                    continue

        st_frame.image(frame, channels="BGR", use_container_width=True)

    cap.release()

# LIVE CAMERA
with tab1:
    st.subheader("International Traffic Feeds")
    # Public Traffic Camera URL (Example: A high-traffic highway feed)
    # Note: These URLs change. I'm using a common test stream format.
    live_url = st.text_input("Enter Camera URL (RTSP/HTTP):",
                             value="https://kamere.mup.gov.rs:4443/Horgos/horgos1.m3u8") # Example NYC/NJ area feed

    if st.button("Start Live Recognition"):
        process_video(live_url)

# TAB 2: UPLOAD
with tab2:
    st.subheader("Historical Video Analysis")
    uploaded_file = st.file_uploader("Upload .mp4 for Analysis", type=["mp4"])
    if uploaded_file:
        with open("temp.mp4", "wb") as f:
            f.write(uploaded_file.read())
        process_video("temp.mp4")

Writing app.py


In [3]:
!pip install -q pyngrok

In [ ]:
import pyngrok.ngrok as ngrok
NGROK_AUTH_TOKEN = ""
!ngrok config add-authtoken {NGROK_AUTH_TOKEN}

# Terminate any existing tunnels
ngrok.kill()

# Open a tunnel to port 8501 (Streamlit's port)
from pyngrok import ngrok
public_url = ngrok.connect(8501, "http")
print(f"🚀 YOUR LIVE APP LINK IS HERE: {public_url.public_url}")

# Run the app with safety flags
!streamlit run app.py --server.port 8501 --server.enableCORS false

Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml
🚀 YOUR LIVE APP LINK IS HERE: https://joya-serous-unartificially.ngrok-free.dev
/bin/bash: line 1: streamlit: command not found


In [5]:
from google.colab import drive
drive.mount('/content/drive')

!cp "/content/drive/My Drive/Car-1000-dataset.zip" /content/local_copy.zip

!zip -T /content/local_copy.zip

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
test of /content/local_copy.zip OK


In [6]:
!unzip /content/local_copy.zip -d /content/dataset

Görüntülenen çıkış son 5000 satıra kısaltıldı.
  inflating: /content/dataset/Car-1000-dataset/all_images/0960/0960_0030.jpg  
  inflating: /content/dataset/Car-1000-dataset/all_images/0960/0960_0031.jpg  
  inflating: /content/dataset/Car-1000-dataset/all_images/0960/0960_0032.jpg  
  inflating: /content/dataset/Car-1000-dataset/all_images/0960/0960_0033.jpg  
  inflating: /content/dataset/Car-1000-dataset/all_images/0960/0960_0034.jpg  
  inflating: /content/dataset/Car-1000-dataset/all_images/0960/0960_0036.jpg  
  inflating: /content/dataset/Car-1000-dataset/all_images/0960/0960_0037.jpg  
  inflating: /content/dataset/Car-1000-dataset/all_images/0960/0960_0038.jpg  
  inflating: /content/dataset/Car-1000-dataset/all_images/0960/0960_0039.jpg  
  inflating: /content/dataset/Car-1000-dataset/all_images/0960/0960_0041.jpg  
  inflating: /content/dataset/Car-1000-dataset/all_images/0960/0960_0042.jpg  
  inflating: /content/dataset/Car-1000-dataset/all_images/0960/0960_0043.jpg  
  inf

In [7]:
import os

# Check the first 10 folders
dataset_path = '/content/dataset/Car-1000-dataset/'
folders = sorted(os.listdir(dataset_path))
print(f"Total Brand Folders: {len(folders)}")
print(f"Sample labels: {folders[:10]}")

# Look inside one folder to see an image
sample_folder = os.path.join(dataset_path, folders[0])
sample_images = os.listdir(sample_folder)
print(f"Sample image in {folders[0]}: {sample_images[0]}")

Total Brand Folders: 3
Sample labels: ['all_images', 'cls_info', 'dataset_split']
Sample image in all_images: 0723


In [8]:
import json
import os

# Path to class_info.JSON
json_path = '/content/dataset/Car-1000-dataset/cls_info/class_info.json'

with open(json_path, 'r', encoding='utf-8') as f:
    class_data = json.load(f)

# The Master Dictionary
id_to_english = {}

for item in class_data:
    raw_id = item['id']

    if "==" in item['name_from_old']:
        # Take the English name of the car
        full_english = item['name_from_old'].split('==')[1]

        clean_name = full_english.replace('_', ' ')

        id_to_english[raw_id] = clean_name
    else:
        # Fallback if '==' is missing
        id_to_english[raw_id] = item['name_from_old'].replace('_', ' ')

# Save the mapping to a file
import pickle
with open('english_mapping.pkl', 'wb') as f:
    pickle.dump(id_to_english, f)

print(f"--- Mapping Complete! {len(id_to_english)} cars processed.")
print("-" * 30)
# Examples to verify
for i, (k, v) in enumerate(id_to_english.items()):
    if i < 10: print(f"ID {k}: {v}")

--- Mapping Complete! 1000 cars processed.
------------------------------
ID 0000: JMC Kaiyun
ID 0001: Farizon Auto Farizon Star H
ID 0002: Dongfeng Fengshen E70
ID 0003: Yutong T7
ID 0004: Caocao 60
ID 0005: BMW M5
ID 0006: Jaguar F-PACE
ID 0007: BYD Dolphin
ID 0008: Audi Audi A7L
ID 0009: Volvo Volvo S90


In [9]:
import os
import shutil
import json

# Dateset split json file
split_json_path = '/content/dataset/Car-1000-dataset/dataset_split/dataset_split.json'
with open(split_json_path, 'r') as f:
    split_data = json.load(f)

base_image_dir = '/content/dataset/Car-1000-dataset/all_images'
output_dir = '/content/organized_data'

# Main directories for train, test and validation
for s in ['train', 'val', 'test']:
    os.makedirs(os.path.join(output_dir, s), exist_ok=True)

# Structured folders
for item in split_data:
    car_id = item['id']

    for split_name in ['train_data', 'val_data', 'test_data']:
        target_folder = split_name.split('_')[0]

        # Create the subfolder for this specific car ID
        os.makedirs(os.path.join(output_dir, target_folder, car_id), exist_ok=True)

        for image_name in item[split_name]:
            src = os.path.join(base_image_dir, car_id, image_name)
            dst = os.path.join(output_dir, target_folder, car_id, image_name)

            if os.path.exists(src):
                shutil.copy(src, dst)

print("Data organized into Train/Val/Test based on the JSON lists!")

Data organized into Train/Val/Test based on the JSON lists!


In [10]:
import tensorflow as tf
from tensorflow.keras import layers, models

# CONFIGURATION
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
DATA_DIR = '/content/organized_data'

# Load Data using the splits
train_ds = tf.keras.utils.image_dataset_from_directory(
    f'{DATA_DIR}/train',
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='int'
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    f'{DATA_DIR}/val',
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='int'
)

# Base Model
base_model = tf.keras.applications.MobileNetV3Large(
    input_shape=(224, 224, 3),
    include_top=False,
    weights='imagenet'
)
base_model.trainable = False

model = models.Sequential([
    layers.Input(shape=(224, 224, 3)),
    # Preprocessing layer specific to MobileNetV3
    layers.Lambda(tf.keras.applications.mobilenet_v3.preprocess_input),
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(512, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(len(train_ds.class_names), activation='softmax')
])

# Compile
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Training with Early Stopping
early_stop = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=3)

print("Starting Training...")
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15,
    callbacks=[early_stop]
)

# Final model
model.save('car_brand_classifier.h5')
print("Training Complete! Model saved as car_brand_classifier.h5")

Found 83756 files belonging to 1000 classes.
Found 27658 files belonging to 1000 classes.
12683000/12683000 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Starting Training...
Epoch 1/15
2618/2618 ━━━━━━━━━━━━━━━━━━━━ 430s 156ms/step - accuracy: 0.0435 - loss: 5.6379 - val_accuracy: 0.1211 - val_loss: 4.6403
Epoch 2/15
2618/2618 ━━━━━━━━━━━━━━━━━━━━ 318s 121ms/step - accuracy: 0.1163 - loss: 4.5549 - val_accuracy: 0.1901 - val_loss: 4.0727
Epoch 3/15
2618/2618 ━━━━━━━━━━━━━━━━━━━━ 319s 122ms/step - accuracy: 0.1598 - loss: 4.1437 - val_accuracy: 0.2218 - val_loss: 3.8196
Epoch 4/15
2618/2618 ━━━━━━━━━━━━━━━━━━━━ 324s 123ms/step - accuracy: 0.1918 - loss: 3.8913 - val_accuracy: 0.2405 - val_loss: 3.6926
Epoch 5/15
2618/2618 ━━━━━━━━━━━━━━━━━━━━ 318s 121ms/step - accuracy: 0.2134 - loss: 3.7298 - val_accuracy: 0.2537 - val_loss: 3.6010
Epoch 6/15
2618/2618 ━━━━━━━━━━━━━━━━━━━━ 325s 123ms/step - accuracy: 0.2301 - loss: 3.6094 - val_accuracy: 0.2568 - val_loss: 3.5616
Epoch 7/15
2618/2618 ━━━━━━━━━━━━━

✅ Training Complete! Model saved as car_brand_classifier.h5
